# Federated AMR — Ceftriaxone × *E. coli*

**Federated learning across 4 hospital sites** (DRIAMS A/B/C/D) using Flower.

| Component | Detail |
|---|---|
| Drug / Pathogen | **Ceftriaxone** / *Escherichia coli* |
| Species | Escherichia coli only (label-stratified split) |
| Architecture | MLP: 6000 → 512 → 256 → 128 → 2 |
| Strategies | FedAvg MLP, FedProx (μ=0.5), FedAvg LR, FedRF tree collection |
| Checkpoints | Per-client + global model saved every round |
| Mixing | Small sites train multiple seeds, average weights (MLP only) |

Flower simulation (Ray backend). Compatible with Google Colab.

Reference implementation for the documentation page
[docs/06b](https://asd-standard.github.io/Dryad_DRIAMS/06b-Ceftriaxone-E-coli/index.html).

## Notes
- **Protocol** — 4 Flower clients, one per DRIAMS site (A/B/C/D); each
  keeps a label-stratified 90/10 split of its own *E. coli* spectra and
  never shares raw data. 30 rounds for the MLP/LR strategies, 10 for the
  FedRF tree collection; the decision thresholds come from the centralized
  tuning.
- **Client mixing** — smaller sites retrain the MLP on several reshuffled
  copies of their local data each round and return the averaged weights
  (`n_mixes()`: 5/4/3 replicas below 500/1500/5000 samples, else 1).
- **Federation is simulated** — server and clients run in one local Ray
  process; FedProx sweeps the mu values in `FEDPROX_MUS` (0.1 and 0.5; the
  table and plot report 0.5) and only model weights or serialised trees cross
  the boundary.
- **Baselines** — centralized MLP/RF on the pooled training frames and
  cross-site A→B/C/D MLP/RF runs; the federated rows in the final table
  report each strategy's best pooled round (round 0 excluded).
- **Checkpoints and outputs** — every round writes per-client and global
  weights under `models/<strategy>/round_XXX/`; metrics/tables/figures go
  to `results/` and the whole run is copied to a numbered `NN-Run` archive
  folder (last cell).

In [ ]:
# Colab dependency install: Flower simulation, MaldiDeepKit/MaldiAMRKit and
# seaborn; a no-op when the packages are already present locally.
!pip install "flwr[simulation]" maldideepkit maldiamrkit seaborn --quiet

In [ ]:
# Mounts Google Drive in Colab; locally the notebook proceeds without it
# and reads the Processed CSVs from the local Dryad checkout.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
# Imports (torch, sklearn, Flower, MaldiDeepKit) and the global seed (42);
# all federated training runs on CPU (FL_DEVICE) because the Ray simulation
# workers are CPU-only.
import warnings, copy, os, io, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     RandomizedSearchCV, cross_val_predict,
                                     StratifiedKFold)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform

import flwr as fl
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

FL_DEVICE = "cpu"
print(f"Flower version: {fl.__version__}")
print(f"FL device: {FL_DEVICE}")

In [ ]:
# Colab/local data root, output folders (results/, models/) and the
# Ceftriaxone x E. coli configuration. The notebook auto-detects the next
# free NN-Run archive folder and copies the run there at the end;
# SITES_PATHS order defines SITE_ORDER A/B/C/D.
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"
DRUG_CSV  = "Ceftriaxone"
SPECIES = "Escherichia coli"

OUT_DIR = Path("./results")
MODEL_DIR = Path("./models")
OUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

# Auto-detect run number
PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06-Ceftazidime-E-coli"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
existing = sorted([int(p.name.split('-')[0]) for p in PROJECT_DIR.glob("[0-9]*-Run")
                   if p.is_dir() and p.name.split('-')[0].isdigit()])
RUN_NUM = (existing[-1] + 1) if existing else 1
DRIVE_RUN_DIR = PROJECT_DIR / f"{RUN_NUM:02d}-Run"
DRIVE_RESULTS = DRIVE_RUN_DIR / "results"
DRIVE_MODELS  = DRIVE_RUN_DIR / "models"
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
print(f"Run #{RUN_NUM} → {DRIVE_RUN_DIR}")

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Drug: {DRUG_NAME}  |  Species: {SPECIES}")

In [ ]:
# Shared constants: threshold sweep, tuning grids, 30 MLP / 10 RF rounds and
# the FedProx mu list. n_mixes() sizes the per-round weight averaging used
# by small sites (5/4/3 local replicas as the client's training set grows,
# else a single pass).
THRESHOLDS = np.linspace(0.05, 0.95, 91)
LR_GRID  = np.linspace(1e-4, 5e-4, 6)
DROP_GRID = np.linspace(0.2, 0.6, 6)
NUM_ROUNDS = 30
NUM_RF_ROUNDS = 10
LOCAL_EPOCHS = 1
BATCH_SIZE = 16
FEDPROX_MUS = [0.1, 0.5]

RF_PARAM_GRID = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [2, 5, 10],
    "class_weight": ["balanced", "balanced_subsample"],
}

def n_mixes(n_train):
    """Number of local training replicas a client averages per round.

    Small clients retrain their model several times per round on reshuffled data
    and average the resulting weights before FedAvg aggregates them; large clients
    train only once.

    Parameters
    ----------
    n_train : int
        Number of training spectra held by the client.

    Returns
    -------
    int
        Replicas per round: 5 (<500 samples), 4 (<1500), 3 (<5000), else 1.
    """
    if n_train < 500: return 5
    if n_train < 1500: return 4
    if n_train < 5000: return 3
    return 1

In [ ]:
# Loads the per-site Ceftriaxone CSVs, keeps Escherichia coli and stacks the
# 6000-bin spectra with the binary resistance label; per-site class counts
# are printed before any splitting.
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    df_eco = df[df["species"] == SPECIES].copy()
    bin_cols = [c for c in df_eco.columns if c.startswith("bin_")]
    X = df_eco[bin_cols].to_numpy(dtype="float32")
    y = df_eco["label"].to_numpy(dtype="int64")
    raw_data[site] = (X, y)
    n_r, n_s = (y == 1).sum(), (y == 0).sum()
    print(f"  Site {site}: {len(y)} samples ({n_s} S, {n_r} R, {n_r/len(y)*100:.1f}% R)")
total = sum(len(raw_data[s][1]) for s in SITE_ORDER)
print(f"\nTotal pooled: {total} samples")

In [ ]:
# Label-stratified 90/10 train/holdout split per site (fixed per-site seeds);
# the holdout is the client's local test set and is never trained on. The
# training frames are concatenated only for the centralized baselines.
client_train = {}; client_test = {}
site_seeds = {"A": 42, "B": 123, "C": 456, "D": 789}
for site in SITE_ORDER:
    X, y = raw_data[site]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.10, stratify=y, random_state=site_seeds[site])
    client_train[site] = (X_tr, y_tr)
    client_test[site] = (X_te, y_te)
    print(f"  Site {site}: train={len(X_tr)}  test={len(X_te)}")

pooled_X_train = np.concatenate([client_train[s][0] for s in SITE_ORDER])
pooled_y_train = np.concatenate([client_train[s][1] for s in SITE_ORDER])
print(f"\nPooled train: {len(pooled_X_train)}")

In [ ]:
# Per-site log1p+standardization fitted on each client's training split
# (kept site-local, mirroring deployment) plus a separate pooled transform
# for the centralized baselines.
client_train_pp = {}; client_test_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
    print(f"  Site {site}: preprocessed")

state_pool = fit_input_transform(pooled_X_train, "log1p+standardize")
X_pool_pp = apply_input_transform(pooled_X_train, state_pool)
y_pool = pooled_y_train.copy()
print(f"\nPooled centralized: {X_pool_pp.shape[0]} preprocessed")

In [ ]:
# Packs the per-site holdouts and their pooled "All" concatenation into the
# dict consumed by every server-side evaluate function.
centralized_test_sets = {}
for site in SITE_ORDER:
    centralized_test_sets[site] = client_test_pp[site]
combined_X_test = np.concatenate([client_test_pp[s][0] for s in SITE_ORDER])
combined_y_test = np.concatenate([client_test_pp[s][1] for s in SITE_ORDER])
centralized_test_sets["All"] = (combined_X_test, combined_y_test)
print(f"Combined test: {combined_X_test.shape[0]} samples")

In [ ]:
# Shared model glue: DatasetFromNumpy, the 6000->512->256->128->2 MLP
# factory, state_dict (de)serialisation for Flower's NumPy parameters and
# the softmax probability adapter.
class DatasetFromNumpy(Dataset):
    """Torch dataset over a NumPy feature/label pair (no extra processing).

    Parameters
    ----------
    X : ndarray (n_samples, 6000)
        Log1p+standardized MALDI-TOF intensities.
    y : ndarray (n_samples,)
        Binary resistance labels (0 susceptible, 1 resistant).
    """
    def __init__(self, X, y):
        """Store the arrays as float32 and int64 tensors."""
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        """Number of spectra in the dataset."""
        return len(self.X)
    def __getitem__(self, i):
        """Spectrum and label pair at index ``i``."""
        return self.X[i], self.y[i]

def build_mlp(dropout_high):
    """Create a fresh binary ``SpectralAttentionMLP`` (6000->512->256->128->2).

    Parameters
    ----------
    dropout_high : float
        Dropout after the projection layer; the low rate is half of it, matching
        the centralized grid search.

    Returns
    -------
    SpectralAttentionMLP
        Untrained model with attention disabled.
    """
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256, 128),
        dropout_high=dropout_high, dropout_low=dropout_high/2.0, use_attention=False)

def model_to_numpy(model):
    """Flatten a model's ``state_dict`` into a list of NumPy arrays.

    The list keeps ``state_dict()`` iteration order, so it round-trips through
    :func:`numpy_to_model`; it is the format Flower's ``NumPyClient`` transports.

    Parameters
    ----------
    model : torch.nn.Module
        Model whose parameters are exported.

    Returns
    -------
    list of ndarray
        One array per state_dict entry (weights and buffers).
    """
    return [v.cpu().numpy() for v in model.state_dict().values()]

def numpy_to_model(model, params):
    """Copy NumPy parameter arrays back into a model's ``state_dict``.

    Inverse of :func:`model_to_numpy`: the arrays are zipped in ``state_dict()``
    order and loaded, so a client adopts the aggregated global weights.

    Parameters
    ----------
    model : torch.nn.Module
        Model updated in place.
    params : list of ndarray
        Parameter arrays in ``state_dict()`` order.
    """
    sd = model.state_dict()
    for k, p in zip(sd.keys(), params): sd[k] = torch.tensor(p)
    model.load_state_dict(sd)

def mlp_predict_proba(model, X_np, dev="cpu"):
    """Resistant-class probability (softmax[:, 1]) for a NumPy batch.

    Parameters
    ----------
    model : torch.nn.Module
        SpectralAttentionMLP (eval mode is set here).
    X_np : ndarray (n_samples, 6000)
        Preprocessed MALDI-TOF spectra.
    dev : str, default="cpu"
        Torch device holding the model.

    Returns
    -------
    ndarray (n_samples,)
        Probability of the resistant class.
    """
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(dev)
    with torch.no_grad():
        return F.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
# Centralized MLP grid search on the pooled training data: 6 lr x 6 dropout
# candidates (AdamW, wd=1e-3, 50 epochs), each scored on a held-out 15%
# split over the shared threshold sweep. The winning (lr, dropout,
# threshold) is reused by every MLP run and by the cross-site MLP baseline.
print("\n=== Centralized MLP Grid Search ===")
X_gs, X_gv, y_gs, y_gv = train_test_split(
    X_pool_pp, y_pool, test_size=0.15, stratify=y_pool, random_state=SEED)
print(f"Grid: train={len(X_gs)}  val={len(X_gv)}")

best_ba, BEST_MLP_LR, BEST_MLP_DH, BEST_MLP_THRESH = -1.0, None, None, 0.5
DEV = "cuda" if torch.cuda.is_available() else "cpu"
for lr in LR_GRID:
    for d in DROP_GRID:
        m = build_mlp(d).to(DEV)
        ds = DatasetFromNumpy(X_gs, y_gs)
        dl = DataLoader(ds, batch_size=64, shuffle=True)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-3)
        crit = nn.CrossEntropyLoss()
        for _ in range(50):
            m.train()
            for xb, yb in dl:
                if len(xb) <= 1: continue
                xb, yb = xb.to(DEV), yb.to(DEV)
                opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        proba = mlp_predict_proba(m, X_gv, dev=DEV)
        for t in THRESHOLDS:
            ba = balanced_accuracy_score(y_gv, proba >= t)
            if ba > best_ba: best_ba = ba; BEST_MLP_LR = lr; BEST_MLP_DH = d; BEST_MLP_THRESH = t
    print(f"  lr={lr:.1e}  best-drop={BEST_MLP_DH:.1f}  BA={best_ba:.4f}")
print(f"\nBest: lr={BEST_MLP_LR:.1e}  dropout={BEST_MLP_DH:.1f}  threshold={BEST_MLP_THRESH:.3f}")

In [ ]:
# Retrains the winning configuration on all pooled data (100 epochs,
# 10-epoch warmup + cosine, patience 3 on the pooled loader loss) and
# evaluates the best checkpoint on each site holdout and the pooled test
# set.
print("\n=== Centralized MLP ===")
cent_m = build_mlp(BEST_MLP_DH).to(DEV)
ds = DatasetFromNumpy(X_pool_pp, y_pool)
dl = DataLoader(ds, batch_size=64, shuffle=True)
opt = torch.optim.AdamW(cent_m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
crit = nn.CrossEntropyLoss()
best_loss, best_sd, patience = float("inf"), None, 0
for ep in range(100):
    cent_m.train()
    for xb, yb in dl:
        if len(xb) <= 1: continue
        xb, yb = xb.to(DEV), yb.to(DEV)
        if ep < 10:
            for pg in opt.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
        opt.zero_grad(); crit(cent_m(xb), yb).backward(); opt.step()
    if ep >= 10: sched.step()
    if ep % 5 == 0:
        cent_m.eval()
        with torch.no_grad():
            vl = sum(crit(cent_m(xb.to(DEV)), yb.to(DEV)).item() for xb, yb in dl) / len(dl)
        if vl < best_loss: best_loss=vl; best_sd={k:v.cpu().clone() for k,v in cent_m.state_dict().items()}; patience=0
        else: patience+=1
        if patience>=3: break
if best_sd: cent_m.load_state_dict(best_sd)
cent_m.eval()

centralized_mlp_results = {}
for site in SITE_ORDER:
    X_tt, y_tt = client_test_pp[site]
    proba = mlp_predict_proba(cent_m, X_tt, dev=DEV)
    centralized_mlp_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    centralized_mlp_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = mlp_predict_proba(cent_m, combined_X_test, dev=DEV)
centralized_mlp_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_MLP_THRESH)
centralized_mlp_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
for site in SITE_ORDER:
    print(f"  {site}: BalAcc={centralized_mlp_results[f'{site}_BalAcc']:.4f}")
print(f"  All: BalAcc={centralized_mlp_results['All_BalAcc']:.4f}")

In [ ]:
# Centralized RF baseline: 3-fold GridSearchCV over the shared grid plus a
# CV-tuned threshold from out-of-fold probabilities; RF_PARAMS and
# BEST_RF_THRESH are reused by the federated and cross-site RF runs.
print("\n=== Centralized RF ===")
grid_rf = GridSearchCV(
    RandomForestClassifier(oob_score=True, random_state=SEED, n_jobs=-1),
    param_grid=RF_PARAM_GRID, cv=3, scoring="balanced_accuracy", n_jobs=-1)
grid_rf.fit(X_pool_pp, y_pool)
rf_cent = grid_rf.best_estimator_
RF_PARAMS = grid_rf.best_params_
print(f"  Best: {RF_PARAMS}")

cv_proba = cross_val_predict(
    RandomForestClassifier(**RF_PARAMS, oob_score=True, random_state=SEED, n_jobs=-1),
    X_pool_pp, y_pool, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
BEST_RF_THRESH = THRESHOLDS[np.argmax([balanced_accuracy_score(y_pool, cv_proba >= t) for t in THRESHOLDS])]
print(f"  Threshold: {BEST_RF_THRESH:.3f}")

centralized_rf_results = {}
for site in SITE_ORDER:
    X_tt, y_tt = client_test_pp[site]
    proba = rf_cent.predict_proba(X_tt)[:, 1]
    centralized_rf_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_RF_THRESH)
    centralized_rf_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = rf_cent.predict_proba(combined_X_test)[:, 1]
centralized_rf_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_RF_THRESH)
centralized_rf_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
for site in SITE_ORDER:
    print(f"  {site}: BalAcc={centralized_rf_results[f'{site}_BalAcc']:.4f}")
print(f"  All: BalAcc={centralized_rf_results['All_BalAcc']:.4f}")

In [ ]:
# Local training for the mixed-weight clients: retrain the model n_mixes
# times on reshuffled copies of the local data, average the resulting
# state_dicts and report the mean loss; the FedProx proximal term is added
# when proximal_mu > 0.
def train_local_mixed(model, X_np, y_np, n_mixes, proximal_mu, global_params):
    """Run one local round, averaging weights over ``n_mixes`` replicas.

    The local data is reshuffled and the model retrained ``n_mixes`` times with
    deterministic seeds; the resulting state_dicts are averaged element-wise and
    loaded back into the model. When ``proximal_mu > 0`` the FedProx penalty
    ``(mu/2) * ||w - w_global||_2`` is added to each replica's local loss.

    Parameters
    ----------
    model : torch.nn.Module
        Client model, updated in place.
    X_np, y_np : ndarray
        Client training spectra (log1p+standardized) and labels.
    n_mixes : int
        Number of shuffled replicas to train and average (see :func:`n_mixes`).
    proximal_mu : float
        FedProx proximal strength; 0.0 = plain FedAvg.
    global_params : list of torch.Tensor or None
        Pre-round global weights for the proximal term.

    Returns
    -------
    float
        Mean loss per replica over the round's batches, proximal term included.
    """
    collect_sds = []; total_loss = 0.0
    for mix_i in range(n_mixes):
        seed = SEED + mix_i * 100 + int(proximal_mu * 1000)
        torch.manual_seed(seed); np.random.seed(seed)
        idx = np.random.permutation(len(X_np))
        X_shuf, y_shuf = X_np[idx], y_np[idx]
        ds = DatasetFromNumpy(X_shuf, y_shuf)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
        opt = torch.optim.AdamW(model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        model.train(); batch_loss = 0.0; n_batch = 0
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if proximal_mu > 0 and global_params is not None:
                prox = sum((w - gw.to(FL_DEVICE)).norm(2) ** 2 for w, gw in zip(model.parameters(), global_params))
                loss = loss + (proximal_mu / 2.0) * prox
            loss.backward(); opt.step()
            batch_loss += loss.item(); n_batch += 1
        total_loss += batch_loss / n_batch
        collect_sds.append({k: v.cpu().clone() for k, v in model.state_dict().items()})
    if len(collect_sds) > 1:
        avg_sd = {}
        for k in collect_sds[0].keys():
            stacked = torch.stack([sd[k].float() for sd in collect_sds])
            avg_sd[k] = stacked.mean(0)
            if k.endswith("num_batches_tracked"):
                avg_sd[k] = avg_sd[k].to(torch.long) if avg_sd[k].dim()==0 else avg_sd[k].round().to(torch.long)
        model.load_state_dict(avg_sd)
    return total_loss / n_mixes

In [ ]:
# Flower client for the MLP strategies: holds one site's training frame,
# sizes its local replica count via n_mixes() and returns the averaged
# weights of its local rounds to the server.
class FedMLPClient(fl.client.NumPyClient):
    """Flower client training the mixed-weight MLP on one DRIAMS site.

    Each round the client retrains its local model ``n_mixes`` times on reshuffled
    data, averages the weights and returns them to FedAvg. There is no
    ``evaluate()``: the server scores the global model on every site holdout.

    Parameters
    ----------
    cid : str or int
        Flower client ID resolved through ``SITE_ORDER``.
    X_train, y_train : ndarray
        Client training spectra (log1p+standardized) and labels.
    """
    def __init__(self, cid, X_train, y_train):
        """Set up the local replica count and a fresh MLP.

        Parameters
        ----------
        cid : str or int
            Flower client ID resolved through ``SITE_ORDER``.
        X_train, y_train : ndarray
            Client training spectra and labels.
        """
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        self.n_mixes = n_mixes(len(X_train))
        self.model = build_mlp(BEST_MLP_DH).to(FL_DEVICE)
    def get_parameters(self, config):
        """Return the client's current weights as NumPy arrays."""
        return model_to_numpy(self.model)
    def set_parameters(self, params):
        """Load the round's global weights into the client model."""
        numpy_to_model(self.model, params)
    def fit(self, parameters, config):
        """Run the client's local round and return the averaged weights.

        Parameters
        ----------
        parameters : list of ndarray
            Global model parameters for this round.
        config : dict
            Round configuration; ``proximal_mu`` switches on the FedProx term.

        Returns
        -------
        tuple
            ``(locally averaged parameters, n_examples, {"train_loss", "n_mixes"})``.
        """
        self.set_parameters(parameters)
        proximal_mu = float(config.get("proximal_mu", 0.0))
        global_copy = None
        if proximal_mu > 0: global_copy = [p.clone().detach() for p in self.model.parameters()]
        loss = train_local_mixed(self.model, self.X_train, self.y_train, self.n_mixes, proximal_mu, global_copy)
        return (self.get_parameters({}), len(self.X_train), {"train_loss": loss, "n_mixes": self.n_mixes})

In [ ]:
# Flower client for the LR strategy: one saga epoch per round on the local
# data, sharing only coef/intercept with the server.
class FedLRClient(fl.client.NumPyClient):
    """Flower client training a warm-started L2 logistic regression.

    Only ``coef_`` and ``intercept_`` cross the client/server boundary; each round
    performs one saga epoch on the client's own data.

    Parameters
    ----------
    cid : str or int
        Flower client ID resolved through ``SITE_ORDER``.
    X_train, y_train : ndarray
        Client training spectra (log1p+standardized) and labels.
    """
    def __init__(self, cid, X_train, y_train):
        """Initialise the client and a zero-weight logistic regression.

        Parameters
        ----------
        cid : str or int
            Flower client ID resolved through ``SITE_ORDER``.
        X_train, y_train : ndarray
            Client training spectra and labels.
        """
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        nf = X_train.shape[1]
        self.model = LogisticRegression(C=1.0, penalty="l2", solver="saga", max_iter=1,
                                        warm_start=True, class_weight="balanced", random_state=SEED)
        self.model.classes_ = np.array([0,1]); self.model.coef_ = np.zeros((1,nf)); self.model.intercept_ = np.zeros(1)
    def get_parameters(self, config):
        """Return ``[coef, intercept]`` as NumPy arrays."""
        return [self.model.coef_.ravel(), self.model.intercept_]
    def set_parameters(self, params):
        """Load the round's global ``[coef, intercept]``.

        Parameters
        ----------
        params : list of ndarray
            ``[coef, intercept]`` aggregated by FedAvg.
        """
        self.model.coef_ = params[0].reshape(1,-1); self.model.intercept_ = params[1]
    def fit(self, parameters, config):
        """Run one saga epoch on the client's data.

        Parameters
        ----------
        parameters : list of ndarray
            Global ``[coef, intercept]`` sent by the server.
        config : dict
            Unused; present in Flower's client API.

        Returns
        -------
        tuple
            ``(updated parameters, n_examples, {"num_examples"})``.
        """
        self.set_parameters(parameters)
        with warnings.catch_warnings(): warnings.simplefilter("ignore"); self.model.fit(self.X_train, self.y_train)
        return (self.get_parameters({}), len(self.X_train), {"num_examples": len(self.X_train)})

In [ ]:
# FedRF tree-collection machinery: joblib (de)serialisation of trees, a
# custom strategy that accumulates (never averages) the trees uploaded by
# the clients, and the client that trains them locally.
def trees_to_array(est):
    """Serialise a list of fitted trees to a compact uint8 array.

    Parameters
    ----------
    est : list of sklearn.tree.DecisionTreeClassifier
        Trees fitted by a client.

    Returns
    -------
    ndarray (n_bytes,) uint8
        joblib bytestream; the only payload Flower has to carry for FedRF.
    """
    buf = io.BytesIO(); joblib.dump(est, buf); buf.seek(0)
    return np.frombuffer(buf.read(), dtype=np.uint8)

def array_to_trees(arr):
    """Deserialise a joblib bytestream back into a list of trees."""
    return joblib.load(io.BytesIO(arr.tobytes()))

class TreeCollectionFedAvg(fl.server.strategy.FedAvg):
    """FedAvg variant that accumulates trees instead of averaging parameters.

    ``aggregate_fit`` appends the trees uploaded by every client to a persistent
    list and returns the whole forest as the global payload, so rounds only ever
    add trees.
    """
    def __init__(self, **kwargs):
        """Initialise the strategy and the persistent tree list."""
        super().__init__(**kwargs)
        self._accumulated_trees = []
    def aggregate_fit(self, server_round, results, failures):
        """Append this round's client trees to the global forest.

        Parameters
        ----------
        server_round : int
            Current round; trees are only ever appended.
        results : list
            ``(ClientProxy, FitRes)`` pairs from the participating clients.
        failures : list
            Failed clients; ignored, so partial participation still aggregates.

        Returns
        -------
        tuple
            ``(Parameters, metrics)`` with ``total_trees`` and the aggregated client
            metrics.
        """
        if not results: return None, {}
        for _, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            if len(ndarrays[0]) > 0: self._accumulated_trees.extend(array_to_trees(ndarrays[0]))
        combined = trees_to_array(self._accumulated_trees)
        aggregated = fl.common.ndarrays_to_parameters([combined])
        metrics = {"total_trees": len(self._accumulated_trees)}
        if self.fit_metrics_aggregation_fn:
            metrics.update(self.fit_metrics_aggregation_fn([fit_res.metrics for _, fit_res in results]))
        return aggregated, metrics

class FedRFClient(fl.client.NumPyClient):
    """Flower client that trains a Random Forest on its own site each round.

    Only the serialised trees are uploaded; the per-round ``random_state``
    combines the seed with the client ID and round number.

    Parameters
    ----------
    cid : str or int
        Flower client ID resolved through ``SITE_ORDER``.
    X_train, y_train : ndarray
        Client training spectra (log1p+standardized) and labels.
    """
    def __init__(self, cid, X_train, y_train):
        """Store the client ID and training frame."""
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
    def get_parameters(self, config):
        """Empty tree payload; trees are uploaded from ``fit()``."""
        return [np.array([], dtype=np.uint8)]
    def fit(self, parameters, config):
        """Train the client's trees for this round and serialise them.

        Parameters
        ----------
        parameters : list of ndarray
            Global tree payload; unused, the client only contributes new trees.
        config : dict
            Round configuration; ``current_round`` seeds the forest.

        Returns
        -------
        tuple
            ``([serialised trees], n_examples, {"n_new_trees", "num_examples"})``.
        """
        server_round = int(config.get("current_round", 0))
        rf = RandomForestClassifier(**RF_PARAMS,
            random_state=SEED + int(self.cid) + 100 * server_round, n_jobs=-1, warm_start=True)
        rf.fit(self.X_train, self.y_train)
        new_trees = list(rf.estimators_)
        return ([trees_to_array(new_trees)], len(self.X_train),
                {"n_new_trees": len(new_trees), "num_examples": len(self.X_train)})

In [ ]:
# Checkpointing wrappers around the four aggregation strategies: before
# delegating to super().aggregate_fit(), each saves every client's local
# parameters for the round plus the aggregated global model under
# models/<strategy>/round_XXX/.
class CheckpointFedAvg(fl.server.strategy.FedAvg):
    """FedAvg strategy that checkpoints client and global MLP weights.

    Each round's client parameters and the aggregated global model are saved to
    ``models/<strategy_name>/round_XXX/`` alongside the normal aggregation.
    """
    def __init__(self, strategy_name, model_dir, **kwargs):
        """Set the strategy name and prepare the checkpoint directory."""
        super().__init__(**kwargs)
        self.strategy_name = strategy_name
        self.model_dir = Path(model_dir) / strategy_name
        self.model_dir.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        """Checkpoint the round's local/global MLP weights and aggregate.

        Every client's parameters are stored as ``client_{cid}.pt`` and the aggregated
        global model as ``global_model.pt``.

        Parameters
        ----------
        server_round : int
            Current federated round.
        results : list
            ``(ClientProxy, FitRes)`` pairs from the participating clients.
        failures : list
            Failed clients (forwarded to super()).

        Returns
        -------
        tuple
            ``(aggregated parameters, aggregated metrics)``.
        """
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(BEST_MLP_DH); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cp.cid}.pt")
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(BEST_MLP_DH); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

class CheckpointFedProx(fl.server.strategy.FedProx):
    """FedProx strategy that checkpoints client and global MLP weights.

    Same checkpoint layout as :class:`CheckpointFedAvg`, with the proximal term
    applied by the clients during ``fit()``.
    """
    def __init__(self, strategy_name, model_dir, **kwargs):
        """Set the strategy name and prepare the checkpoint directory."""
        super().__init__(**kwargs)
        self.strategy_name = strategy_name
        self.model_dir = Path(model_dir) / strategy_name
        self.model_dir.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        """Checkpoint the round's local/global MLP weights, then aggregate.

        Every client's parameters are stored as ``client_{cid}.pt`` and the aggregated
        global model as ``global_model.pt``; the proximal term was applied locally by
        the FedProx clients.

        Parameters
        ----------
        server_round : int
            Current federated round.
        results : list
            ``(ClientProxy, FitRes)`` pairs from the participating clients.
        failures : list
            Failed clients (forwarded to super()).

        Returns
        -------
        tuple
            ``(aggregated parameters, aggregated metrics)``.
        """
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(BEST_MLP_DH); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cp.cid}.pt")
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(BEST_MLP_DH); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

class CheckpointLRFedAvg(fl.server.strategy.FedAvg):
    """FedAvg strategy that checkpoints client and global LR parameters.

    ``coef``/``intercept`` are saved as ``.npz`` per client per round, plus the
    aggregated global model.
    """
    def __init__(self, model_dir, **kwargs):
        """Prepare the checkpoint directory for the federated LR."""
        super().__init__(**kwargs)
        self.model_dir = Path(model_dir) / "fedavg_lr"
        self.model_dir.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        """Checkpoint the round's local/global LR arrays and aggregate.

        Each client's ``[coef, intercept]`` is stored as ``client_{cid}.npz`` and the
        aggregated model as ``global_model.npz``.

        Parameters
        ----------
        server_round : int
            Current federated round.
        results : list
            ``(ClientProxy, FitRes)`` pairs from the participating clients.
        failures : list
            Failed clients (forwarded to super()).

        Returns
        -------
        tuple
            ``(aggregated parameters, aggregated metrics)``.
        """
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            np.savez(round_dir / f"client_{cp.cid}.npz", coef=ndarrays[0], intercept=ndarrays[1])
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            np.savez(round_dir / "global_model.npz", coef=nd[0], intercept=nd[1])
        return aggregated, metrics

class CheckpointTreeCollection(TreeCollectionFedAvg):
    """Tree-collection strategy that checkpoints the FedRF payloads.

    Client tree arrays are saved as ``client_{cid}_trees.npy`` and the accumulated
    forest as ``global_trees.npy`` before/after aggregation.
    """
    def __init__(self, model_dir, **kwargs):
        """Prepare the checkpoint directory for the tree collection."""
        super().__init__(**kwargs)
        self.model_dir = Path(model_dir) / "fedrf"
        self.model_dir.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        """Checkpoint this round's client trees, then accumulate them.

        Client payloads are stored as ``client_{cid}_trees.npy``; the accumulated
        forest returned by super() is stored as ``global_trees.npy``.

        Parameters
        ----------
        server_round : int
            Current federated round.
        results : list
            ``(ClientProxy, FitRes)`` pairs from the participating clients.
        failures : list
            Failed clients (forwarded to super()).

        Returns
        -------
        tuple
            ``(aggregated parameters, aggregated metrics)``.
        """
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            if len(ndarrays[0]) > 0: np.save(round_dir / f"client_{cp.cid}_trees.npy", ndarrays[0])
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            np.save(round_dir / "global_trees.npy", nd[0])
        return aggregated, metrics

In [ ]:
# Server-side evaluation factories for the three model families: rebuild the
# current global model from the round's parameters, score each site holdout
# and their pooled concatenation, and append the record to the history list
# driving the plots and CSVs.
def get_mlp_eval_fn(test_dict, threshold, hist_list):
    """Build the MLP server-side evaluation callback.

    Parameters
    ----------
    test_dict : dict
        ``{site: (X_test, y_test)}`` per-site holdouts.
    threshold : float
        Decision threshold for the resistant class.
    hist_list : list
        Per-round history to append records to.

    Returns
    -------
    callable
        ``evaluate(server_round, parameters, config)`` returning
        ``(1 - pooled BalAcc, record)``.
    """
    def evaluate(server_round, parameters, config):
        """Evaluate the global MLP on every site holdout for one round.

        Parameters
        ----------
        server_round : int
            Current federated round (1-based).
        parameters : list of ndarray
            Aggregated global parameters.
        config : dict
            Unused; Flower passes the round configuration.

        Returns
        -------
        tuple
            ``(loss, record)`` with ``loss = 1 - All_BalAcc``; the record is appended
            to ``hist_list``.
        """
        m = build_mlp(BEST_MLP_DH); numpy_to_model(m, parameters); m.to(FL_DEVICE); m.eval()
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = mlp_predict_proba(m, X_tt, dev=FL_DEVICE)
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

def get_lr_eval_fn(test_dict, threshold, hist_list):
    """Build the server-side evaluation callback for the LR strategy.

    The LR is rebuilt from the aggregated ``[coef, intercept]`` pair for scoring
    (lbfgs, class_weight balanced).

    Parameters
    ----------
    test_dict : dict
        ``{site: (X_test, y_test)}`` per-site holdouts.
    threshold : float
        Decision threshold for the resistant class.
    hist_list : list
        Per-round history to append records to.

    Returns
    -------
    callable
        ``evaluate(server_round, parameters, config)`` returning
        ``(1 - pooled BalAcc, record)``.
    """
    def evaluate(server_round, parameters, config):
        """Evaluate the global LR on every site holdout for one round.

        Parameters
        ----------
        server_round : int
            Current federated round (1-based).
        parameters : list of ndarray
            Aggregated ``[coef, intercept]``.
        config : dict
            Unused; Flower passes the round configuration.

        Returns
        -------
        tuple
            ``(loss, record)`` with ``loss = 1 - All_BalAcc``; the record is appended
            to ``hist_list``.
        """
        lr = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000)
        lr.classes_ = np.array([0,1]); lr.coef_ = parameters[0].reshape(1,-1); lr.intercept_ = parameters[1]
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = lr.predict_proba(X_tt)[:, 1]; preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

def get_rf_eval_fn(test_dict, threshold, hist_list):
    """Build the server-side evaluation callback for the tree collection.

    The global forest is rebuilt from the aggregated uint8 tree payload; an empty
    payload yields a neutral 0.5 score.

    Parameters
    ----------
    test_dict : dict
        ``{site: (X_test, y_test)}`` per-site holdouts.
    threshold : float
        Decision threshold (the CV-tuned ``BEST_RF_THRESH``).
    hist_list : list
        Per-round history to append records to.

    Returns
    -------
    callable
        ``evaluate(server_round, parameters, config)`` returning
        ``(1 - pooled BalAcc, record)``.
    """
    def evaluate(server_round, parameters, config):
        """Evaluate the global forest on every site holdout for one round.

        Parameters
        ----------
        server_round : int
            Current federated round (1-based).
        parameters : list of ndarray
            Aggregated uint8 tree payload (empty before the first fit).
        config : dict
            Unused; Flower passes the round configuration.

        Returns
        -------
        tuple
            ``(loss, record)`` with ``loss = 1 - All_BalAcc``; the record includes
            ``n_trees`` and is appended to ``hist_list``.
        """
        if len(parameters[0]) == 0: return (1.0, {"All_BalAcc": 0.5})
        trees = array_to_trees(parameters[0])
        rf = RandomForestClassifier(**RF_PARAMS, n_jobs=-1)
        rf.estimators_ = trees; rf.n_classes_ = 2; rf.classes_ = np.array([0,1]); rf.n_outputs_ = 1
        record = {"round": server_round, "n_trees": len(trees)}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = rf.predict_proba(X_tt)[:, 1]; preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

In [ ]:
# Flower client factories mapping integer client IDs 0-3 onto DRIAMS
# sites A-D for the MLP, LR and RF strategies.
def mlp_client_fn(cid):
    """Instantiate the FedMLPClient for one simulated site.

    Parameters
    ----------
    cid : str
        Flower client ID, resolved through ``SITE_ORDER``.

    Returns
    -------
    fl.client.Client
        MLP client for that site's preprocessed frames.
    """
    site = SITE_ORDER[int(cid)]
    return FedMLPClient(cid, *client_train_pp[site]).to_client()

def lr_client_fn(cid):
    """Instantiate the FedLRClient for one simulated site.

    Parameters
    ----------
    cid : str
        Flower client ID, resolved through ``SITE_ORDER``.

    Returns
    -------
    fl.client.Client
        LR client for that site's preprocessed frames.
    """
    site = SITE_ORDER[int(cid)]
    return FedLRClient(cid, *client_train_pp[site]).to_client()

def rf_client_fn(cid):
    """Instantiate the FedRFClient for one simulated site.

    Parameters
    ----------
    cid : str
        Flower client ID, resolved through ``SITE_ORDER``.

    Returns
    -------
    fl.client.Client
        RF client for that site's preprocessed frames.
    """
    site = SITE_ORDER[int(cid)]
    return FedRFClient(cid, *client_train_pp[site]).to_client()

In [ ]:
# FedAvg MLP run: 30 rounds, one local round per client with its n_mixes
# replicas, per-round checkpoints and server-side evaluation.
print("\n=== FedAvg MLP ===")
eval_hist_fedavg = []
_eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedavg)
strategy = CheckpointFedAvg("fedavg_mlp", MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp(BEST_MLP_DH))))
strategy.model_dir = Path(MODEL_DIR) / "fedavg_mlp"
fl.simulation.start_simulation(
    client_fn=mlp_client_fn, num_clients=4, config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})
fedavg_hist = eval_hist_fedavg.copy()
print(f"FedAvg MLP done. {len(fedavg_hist)} rounds.")

In [ ]:
# FedProx MLP runs over FEDPROX_MUS: same federation as FedAvg plus the
# proximal term in each client's local loss; every mu gets its own
# checkpoint folder and history.
fedprox_histories = {}
for mu in FEDPROX_MUS:
    print(f"\n=== FedProx MLP (mu={mu}) ===")
    eval_hist_fedprox = []
    _eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedprox)
    sname = f"fedprox_mlp_mu{mu}"
    strategy = CheckpointFedProx(sname, MODEL_DIR,
        fraction_fit=1.0, fraction_evaluate=0.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        proximal_mu=mu, evaluate_fn=_eval_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp(BEST_MLP_DH))))
    strategy.model_dir = Path(MODEL_DIR) / sname
    fl.simulation.start_simulation(
        client_fn=mlp_client_fn, num_clients=4, config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})
    fedprox_histories[mu] = eval_hist_fedprox.copy()
    print(f"FedProx mu={mu} done. {len(eval_hist_fedprox)} rounds.")

In [ ]:
# FedAvg on logistic regression: clients send a single saga epoch per round
# (coef + intercept only); the LR evaluate function rebuilds the model from
# the aggregated arrays.
print("\n=== FedAvg LR ===")
eval_hist_fedlr = []
_eval_fn = get_lr_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedlr)
nr_feat = client_train_pp["A"][0].shape[1]
lr_init = LogisticRegression(C=1.0, penalty="l2", solver="saga", max_iter=1,
                             warm_start=True, class_weight="balanced", random_state=SEED)
lr_init.classes_ = np.array([0,1]); lr_init.coef_ = np.zeros((1,nr_feat)); lr_init.intercept_ = np.zeros(1)
init_params = [lr_init.coef_.ravel(), lr_init.intercept_]
strategy = CheckpointLRFedAvg(MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(init_params))
strategy.model_dir = Path(MODEL_DIR) / "fedavg_lr"
fl.simulation.start_simulation(
    client_fn=lr_client_fn, num_clients=4, config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})
fedlr_hist = eval_hist_fedlr.copy()
print(f"FedAvg LR done. {len(fedlr_hist)} rounds.")

In [ ]:
# FedRF run: 10 tree-collection rounds, each client contributing a fresh
# Random Forest that the custom strategy appends to the global payload.
print("\n=== FedRF (Tree Collection) ===")
eval_hist_fedrf = []
_eval_fn = get_rf_eval_fn(client_test_pp, BEST_RF_THRESH, eval_hist_fedrf)
strategy = CheckpointTreeCollection(MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([np.array([], dtype=np.uint8)]))
strategy.model_dir = Path(MODEL_DIR) / "fedrf"
fl.simulation.start_simulation(
    client_fn=rf_client_fn, num_clients=4, config=fl.server.ServerConfig(num_rounds=NUM_RF_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})
fedrf_hist = eval_hist_fedrf.copy()
print(f"FedRF done. {len(fedrf_hist)} rounds.")

In [ ]:
# Cross-site baseline: train on DRIAMS-A only (A's own transform), test on
# A and B/C/D with the shared MLP configuration - the no-collaboration
# lower bound for the federated runs.
print("\n=== Cross-Site MLP (A -> B/C/D) ===")
X_A_train, y_A_train = client_train["A"]
state_cs = fit_input_transform(X_A_train, "log1p+standardize")
X_A_tr = apply_input_transform(X_A_train, state_cs)
cross_test_sets = {}
for sk in "BCD":
    X_te, y_te = client_test[sk]
    cross_test_sets[sk] = (apply_input_transform(X_te, state_cs), y_te)
cross_test_sets["A"] = (apply_input_transform(client_test["A"][0], state_cs), client_test["A"][1])
cs_m = build_mlp(BEST_MLP_DH).to(DEV)
ds_cs = DatasetFromNumpy(X_A_tr, y_A_train)
dl_cs = DataLoader(ds_cs, batch_size=64, shuffle=True)
opt_cs = torch.optim.AdamW(cs_m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched_cs = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cs, T_max=90, eta_min=1e-6)
crit_cs = nn.CrossEntropyLoss()
for ep in range(100):
    cs_m.train()
    for xb, yb in dl_cs:
        if len(xb) <= 1: continue
        xb, yb = xb.to(DEV), yb.to(DEV)
        if ep < 10:
            for pg in opt_cs.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
        opt_cs.zero_grad(); crit_cs(cs_m(xb), yb).backward(); opt_cs.step()
    if ep >= 10: sched_cs.step()
cs_m.eval()
cross_site_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = mlp_predict_proba(cs_m, X_tt, dev=DEV)
    cross_site_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    cross_site_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  {name}: BalAcc={cross_site_results[f'{name}_BalAcc']:.4f}  AUC={cross_site_results[f'{name}_AUC']:.4f}")
X_BCD = np.concatenate([cross_test_sets[s][0] for s in "BCD"])
y_BCD = np.concatenate([cross_test_sets[s][1] for s in "BCD"])
proba_bcd = mlp_predict_proba(cs_m, X_BCD, dev=DEV)
cross_site_results["All_BalAcc"] = balanced_accuracy_score(y_BCD, proba_bcd >= BEST_MLP_THRESH)
cross_site_results["All_AUC"] = roc_auc_score(y_BCD, proba_bcd)
print(f"  All (B+C+D): BalAcc={cross_site_results['All_BalAcc']:.4f}  AUC={cross_site_results['All_AUC']:.4f}")

In [ ]:
# Cross-site RF counterpart: same A-trained transform and tuned RF, scored
# on A/B/C/D at the CV-tuned threshold.
print("\n=== Cross-Site RF (A -> B/C/D) ===")
rf_cs = RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1)
rf_cs.fit(X_A_tr, y_A_train)
cross_site_rf_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = rf_cs.predict_proba(X_tt)[:, 1]
    cross_site_rf_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_RF_THRESH)
    cross_site_rf_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  {name}: BalAcc={cross_site_rf_results[f'{name}_BalAcc']:.4f}")

In [ ]:
# Builds the final results table: centralized and cross-site methods are
# read from their result dicts, the federated strategies contribute their
# best pooled round (round 0 excluded), and each row carries a compact
# label with the peak round.
def best_metrics(h):
    """Best pooled round of a federated history (round 0 excluded).

    Parameters
    ----------
    h : list of dict
        Per-round records with ``All_BalAcc`` and ``round`` keys.

    Returns
    -------
    tuple
        ``(best_record, best_round)``; ``({}, 0)`` for an empty history.
    """
    if not h: return {}, 0
    skip0 = h[1:]
    best = max(skip0, key=lambda r: r.get("All_BalAcc", 0))
    return best, int(best.get("round", 0))

rows = []
def add_row(method, cs_source=None, fed_hist=None):
    """Append one method's row to the results table.

    Parameters
    ----------
    method : str
        Display name of the model/strategy.
    cs_source : dict, optional
        Fixed metrics dict for centralized/cross-site methods.
    fed_hist : list of dict, optional
        Per-round history for federated methods; the best pooled round (round 0
        excluded) supplies the per-site metrics.
    """
    r = {"Method": method}; peak_round = ""
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", np.nan)
        elif fed_hist is not None:
            lm, pr = best_metrics(fed_hist)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
            peak_round = f" (r{pr})"
    if fed_hist is not None:
        lm, pr = best_metrics(fed_hist)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan)
        r["All_AUC"] = lm.get("All_AUC", np.nan)
        r["Peak_Round"] = int(pr)
    elif cs_source:
        r["All_BalAcc"] = cs_source.get("All_BalAcc", np.nan)
        r["All_AUC"] = cs_source.get("All_AUC", np.nan)
        r["Peak_Round"] = 0
    r["Label"] = method + peak_round
    rows.append(r)

add_row("Centralized MLP", cs_source=centralized_mlp_results)
add_row("Centralized RF", cs_source=centralized_rf_results)
add_row("FL FedAvg (MLP)", fed_hist=fedavg_hist)
add_row("FL FedProx mu=0.5 (MLP)", fed_hist=fedprox_histories[0.5])
add_row("FL FedAvg (LR)", fed_hist=fedlr_hist)
add_row("FL FedRF (Trees)", fed_hist=fedrf_hist)
add_row("Cross-Site MLP", cs_source=cross_site_results)
add_row("Cross-Site RF", cs_source=cross_site_rf_results)

df_results = pd.DataFrame(rows)
cols = ["Method", "Label", "Peak_Round"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
# Convergence figure: left = FedAvg per-site vs pooled Balanced Accuracy;
# right = the four federated strategies (pooled) with peak rounds marked and
# the centralized MLP/RF levels as horizontal references. Saved as
# convergence.pdf.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for site in SITE_ORDER:
    vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedavg_hist]
    ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
vals_all = [h.get("All_BalAcc", np.nan) for h in fedavg_hist]
ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.set_title("FedAvg MLP"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[1]
plots = [("FedAvg MLP", fedavg_hist, "#ff7f0e", "-"),
         ("FedProx mu=0.5", fedprox_histories[0.5], "#d62728", "--"),
         ("FedAvg LR", fedlr_hist, "#1f77b4", "-."),
         ("FedRF", fedrf_hist, "#2ca02c", "-")]
for label, hist, c, ls in plots:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
    skip0 = hist[1:]
    if skip0:
        best = max(skip0, key=lambda h: h.get("All_BalAcc", 0))
        pr = int(best.get("round", 0))
        ax.axvline(pr, color=c, ls=':', alpha=0.4, lw=1)
        ax.annotate(f"r{pr}", (pr, best.get("All_BalAcc", 0)), textcoords="offset points", xytext=(3,5), fontsize=7, color=c)
ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2, label='Centralized MLP')
ax.axhline(centralized_rf_results["All_BalAcc"], color='gray', ls='--', lw=2, label='Centralized RF')
ax.set_title("All Methods (peak rounds marked)"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)
fig.suptitle(f"{DRUG_NAME} x {SPECIES} — FL Convergence", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# ══ FedProx μ retry (set RETRY_MU, re-run, then re-run plots) ══
# Interactive retry utility: set RETRY_MU to a value not already in
# fedprox_histories and re-run to add another proximal run, then re-run the
# convergence/heatmap cells. Left as None, the cell is a no-op.
RETRY_MU = None  # <-- CHANGE THIS to e.g. 0.1, 0.3, then re-run this cell
if RETRY_MU is None:
    print("Set RETRY_MU to a value (e.g. 0.1), then re-run this cell.")
    print("After running, re-run the convergence + heatmap cells.")
elif RETRY_MU in fedprox_histories:
    print(f"mu={RETRY_MU} already exists. Skipping.")
else:
    print(f"\n=== Re-run FedProx MLP (mu={RETRY_MU}) ===")
    eval_hist_retry = []
    _eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_retry)
    strategy = CheckpointFedProx(f"fedprox_mlp_mu{RETRY_MU}", MODEL_DIR,
        fraction_fit=1.0, fraction_evaluate=0.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        proximal_mu=RETRY_MU, evaluate_fn=_eval_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp(BEST_MLP_DH))))
    strategy.model_dir = Path(MODEL_DIR) / f"fedprox_mlp_mu{RETRY_MU}"
    for rnd in range(1, NUM_ROUNDS+1): strategy.model_dir / f"round_{rnd:03d}"
    fl.simulation.start_simulation(
        client_fn=mlp_client_fn, num_clients=4,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})
    fedprox_histories[RETRY_MU] = eval_hist_retry.copy()
    print(f"FedProx mu={RETRY_MU} done. {len(eval_hist_retry)} rounds.")
    last = eval_hist_retry[-1] if eval_hist_retry else {}
    print(f"  Final All_BalAcc: {last.get('All_BalAcc', np.nan):.4f}")
    print(f"  Best All_BalAcc:  {max((h.get('All_BalAcc',0) for h in eval_hist_retry[1:]), default=0):.4f}")
    print("\nRe-run convergence + heatmap cells to see updated results.")

In [ ]:
# Final Balanced Accuracy heatmap for every method row; saved as
# heatmap_balacc.pdf.
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Label"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)): ba_data[r["Label"]]["All"] = r["All_BalAcc"]
df_ba_hm = pd.DataFrame(ba_data).T
df_ba_hm = df_ba_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_ba_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba_hm)*0.5)))
sns.heatmap(df_ba_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — Balanced Accuracy", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Final AUC-ROC heatmap for every method row; saved as heatmap_auc.pdf.
auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Label"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)): auc_data[r["Label"]]["All"] = r["All_AUC"]
df_auc_hm = pd.DataFrame(auc_data).T
df_auc_hm = df_auc_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_auc_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc_hm)*0.5)))
sns.heatmap(df_auc_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — AUC-ROC", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Persists the per-strategy round histories, the final table and the tuning
# values used (best_params_used.txt) under results/.
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
pd.DataFrame(fedavg_hist).to_csv(OUT_DIR / "fedavg_per_round.csv", index=False)
pd.DataFrame(fedlr_hist).to_csv(OUT_DIR / "fedlr_per_round.csv", index=False)
pd.DataFrame(fedrf_hist).to_csv(OUT_DIR / "fedrf_per_round.csv", index=False)
for mu in FEDPROX_MUS:
    pd.DataFrame(fedprox_histories[mu]).to_csv(OUT_DIR / f"fedprox_mu{mu}_per_round.csv", index=False)
with open(OUT_DIR / "best_params_used.txt", "w") as f:
    f.write(f"DRUG={DRUG_NAME}\n")
    f.write(f"BEST_MLP_LR={BEST_MLP_LR}\nBEST_MLP_DH={BEST_MLP_DH}\nBEST_MLP_THRESH={BEST_MLP_THRESH}\n")
    f.write(f"RF_PARAMS={RF_PARAMS}\nBEST_RF_THRESH={BEST_RF_THRESH}\n")
print("\n" + "="*60)
print(f"  Done. Results in {OUT_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")): print(f"    {f.name}")

In [ ]:
# Archives the run: copies this notebook, the results/ files and the models/
# checkpoints into the auto-numbered NN-Run folder, so run outputs live
# outside the repo working tree.
import shutil
nb_source = Path("06-03b-Ceftriaxone-E-coli-Federated.ipynb")
if not nb_source.exists(): nb_source = Path.cwd() / "06-03b-Ceftriaxone-E-coli-Federated.ipynb"
if not nb_source.exists():
    import glob as _g
    candidates = list(_g.glob("/content/**/06-03b*.ipynb", recursive=True))
    if candidates: nb_source = Path(candidates[0])
if nb_source.exists(): shutil.copy(str(nb_source), str(DRIVE_RESULTS / "notebook.ipynb"))
for f in OUT_DIR.glob("*"):
    if f.is_file(): shutil.copy2(str(f), str(DRIVE_RESULTS / f.name))
for d in MODEL_DIR.glob("*"):
    if d.is_dir():
        dst = DRIVE_MODELS / d.name
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(str(d), str(dst))
print(f"Run #{RUN_NUM} archived to {DRIVE_RUN_DIR}")

---
**Done.** Federated analysis for **Ceftriaxone × Escherichia coli** complete. Run → `06-Ceftazidime-E-coli/{RUN_NUM:02d}-Run/`.